# 2. 음성 분리 노트북 (A4, A5 공격자용)

SepFormer(`speechbrain/sepformer-whamr16k`)로 경쟁 화자 1, 2, 4명과 다화자 잡음(8명) 조건의 잡음 혼합 음성을 두 출력으로 분리합니다.

- 먼저 `01_main_experiment` 노트북의 0번 셀을 실행해 두어야 합니다. 이 노트북은 그 설정(`runs/active_run.json`)을 그대로 읽습니다.
- `01_main_experiment` 노트북의 커널을 끈 상태에서 실행하세요(그래픽카드 메모리 부족 방지).
- 끝나면 이 노트북의 커널을 끄고 `01_main_experiment`의 4번 셀로 돌아갑니다.
- 중간에 끊겨도 두 셀을 다시 실행하면 이어서 진행합니다.
- PyTorch와 SpeechBrain 설치는 README의 '설치'를 참고하세요.

In [ ]:
# ===== 음성 분리 준비 =====
import json
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from tqdm import tqdm
from speechbrain.inference.separation import SepformerSeparation
from speechbrain.utils.fetching import LocalStrategy

cfg = json.loads((Path.cwd() / "runs" / "active_run.json").read_text(encoding="utf-8"))
ROOT, RUN = Path(cfg["ROOT"]), Path(cfg["RUN"])
SEP_NOISES, ADAPT_SNRS = cfg["SEP_NOISES"], cfg["ADAPT_SNRS"]
SEPARATED = RUN / "separated"
print("저장 위치:", RUN.relative_to(ROOT).as_posix())

sep_model = SepformerSeparation.from_hparams(
    source="speechbrain/sepformer-whamr16k", # 잡음, 잔향이 섞인 두 화자 음성(WHAMR!)으로 학습된 모델
    savedir=str(ROOT / "models/sepformer-whamr16k"),
    local_strategy=LocalStrategy.COPY, # Windows 바로가기(symlink) 오류 방지
    run_opts={"device": "cuda:0"})
print("torch:", torch.__version__, "| 모델 샘플링 주파수:", sep_model.hparams.sample_rate)

In [ ]:
# ===== 음성 분리 실행 =====
mix_df = pd.read_csv(RUN / "mixed.csv", dtype={"answer": str})
out = RUN / "separated.csv"
done = set(pd.read_csv(out, usecols=["key"])["key"]) if out.exists() else set()
todo = mix_df[mix_df.noise.isin(SEP_NOISES) & mix_df.snr.isin(ADAPT_SNRS) & ~mix_df.key.isin(done)]
print(f"[음성 분리] 완료 {len(done)} | 남음 {len(todo)}")

def save(rows):
    if rows:
        pd.DataFrame(rows).to_csv(out, mode="a", header=not out.exists(), index=False)

buf = []
for r in tqdm(todo.itertuples(), total=len(todo)):
    mix, fs = sf.read(ROOT / r.path, dtype="float32")
    with torch.inference_mode():
        est = sep_model.separate_batch(torch.from_numpy(mix).unsqueeze(0))
    est = est[0].cpu().numpy()
    folder = SEPARATED / r.noise / f"snr{r.snr}"
    folder.mkdir(parents=True, exist_ok=True)
    paths = []
    for k in range(est.shape[1]):
        y = est[:, k]
        y = y / max(np.abs(y).max(), 1e-9) * 0.5 # 음량 맞추기 (최대치 0.5)
        p = folder / f"{r.clip_id}_s{k}.wav"
        sf.write(p, y, fs)
        paths.append(p.relative_to(ROOT).as_posix())
    buf.append({"key": r.key, "clip_id": r.clip_id, "answer": r.answer, "noise": r.noise,
                "snr": r.snr, "path_s0": paths[0], "path_s1": paths[1]})
    if len(buf) >= 50:
        save(buf)
        buf = []
save(buf)
print("음성 분리 완료:", len(pd.read_csv(out)), "개")


[음성 분리] 완료 0 | 남음 4800


100%|██████████████████████████████████████████████████████████████████████████████| 4800/4800 [25:49<00:00,  3.10it/s]

음성 분리 완료: 4800 개
